# Módulo 07 - Módulos e Pacotes

---

Até aqui, todo o código que você escreveu viveu dentro de um único notebook. Neste módulo você vai aprender a **reaproveitar código entre notebooks diferentes**, começando pelas formas de trazer código pronto para o seu programa com `import`, `from` e `as`. Em seguida, você vai criar o seu primeiro **módulo** — um arquivo `.py` com uma classe que lê arquivos CSV — e depois organizar módulos relacionados dentro de um **pacote**, uma pasta que agrupa arquivos que trabalham juntos. Por fim, você vai conhecer o **PyPI** e o **pip**, as ferramentas que dão acesso a milhares de pacotes criados pela comunidade Python, e vai usar o pacote `requests` para consultar um serviço na internet. Ao final deste módulo, você vai entender como o próprio Python — e bibliotecas como Pandas e NumPy, que vêm a seguir no curso — são organizados por dentro.

Curso: Ready To Deploy

Criado por: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Tópicos

| **Tópico** | Descrição |
| --- | --- |
| 1. from / import / as | Como importar um módulo inteiro, apenas alguns nomes dele, ou apelidá-los. |
| 2. Módulos | O que é um módulo, como criar o seu próprio e importar código reutilizável entre notebooks. |
| 3. Pacotes | Como organizar vários módulos relacionados dentro de uma pasta (pacote) e importar código de dentro dela. |
| 4. Baixando pacotes | Como encontrar, instalar e usar pacotes de terceiros com PyPI e pip, aplicado a uma consulta HTTP com o pacote requests. |

---

## 1. from / import / as

O Python vem com uma **biblioteca padrão** enorme — a chamada filosofia "*baterias inclusas*" — cheia de módulos prontos para tarefas comuns (veja a lista completa [aqui](https://docs.python.org/3/library/)). Para usar qualquer um deles, primeiro é preciso **importá-lo**.

### 1.1 import

A forma mais simples é `import nome_do_modulo`, que traz o módulo **inteiro**. Depois, cada função ou variável dele é acessada com `nome_do_modulo.nome_do_item`.

**Exemplo:** o módulo `random`, para gerar valores e escolhas aleatórias.

In [1]:
import random

In [2]:
opcoes = ['pedra', 'papel', 'tesoura']

escolha_do_computador = random.choice(opcoes)
print(escolha_do_computador)

tesoura


In [3]:
numero_aleatorio = random.random()  # float aleatório no intervalo [0, 1)
print(numero_aleatorio)

0.5191044337047542


**Exemplo:** o módulo `math`, com funções e constantes matemáticas.

In [4]:
import math

In [5]:
potencia = math.pow(10, 3)
print(potencia)

1000.0


In [6]:
# math.ceil() sempre arredonda PARA CIMA, diferente do round() visto no Módulo 01
numero_arredondado = math.ceil(10.1)
print(numero_arredondado)

11


In [7]:
print(math.pi)

3.141592653589793


### 1.2 from / import

Quando só precisamos de um ou alguns itens específicos de um módulo, importamos exatamente esses nomes com `from modulo import nome`. A vantagem é usar o nome diretamente, sem o prefixo do módulo.

```python
from modulo import nome
```

> ⚠️ **Atenção:** só o que foi explicitamente importado fica disponível. Se o módulo `time` tem as funções `time()` e `sleep()`, mas importamos apenas `time`, tentar usar `sleep()` gera um `NameError` (visto no Módulo 03) — ela simplesmente não existe no notebook.

In [8]:
from time import time

# 'time' foi importado e funciona
print(time())  # segundos desde 01/01/1970 (a "época Unix")

# sleep(1)  # ❌ NameError: name 'sleep' is not defined (sleep não foi importado)

1789135666.9954908


Para importar mais de um nome do mesmo módulo, separamos por vírgula:

In [9]:
from time import time, sleep

instante_inicial = time()
sleep(1)  # pausa a execução do programa por 1 segundo
instante_final = time()

print(f'Tempo decorrido: {instante_final - instante_inicial:.2f} segundos')

Tempo decorrido: 1.00 segundos


### 1.3 from / import / as

Podemos apelidar (renomear) um módulo ou um item importado com `as`. É útil para encurtar nomes longos ou evitar conflitos com outros nomes já usados no código.

**Exemplo:** a classe `datetime`, que mora dentro do módulo de mesmo nome, `datetime` — apelidada de `dt` para não confundir os dois.

In [10]:
from datetime import datetime as dt

In [11]:
print(dt.now())

2026-09-11 11:07:48.102980


In [12]:
print(dt.now().day)

11


In [13]:
print(dt.now().year)

2026


> 💡 **Dica:** apelidos curtos e consagrados pelo mercado — como `import pandas as pd` e `import numpy as np` — são uma convenção que você vai ver em praticamente todo código de análise de dados em Python, inclusive nos próximos módulos deste curso.

---

## 2. Módulos

Um **módulo** é simplesmente um arquivo `.py` contendo código Python — funções, classes, variáveis — pronto para ser reaproveitado. Os módulos importados na seção 1 fazem parte da biblioteca padrão do Python; nesta seção você vai criar o seu próprio.

### 2.1 Motivação

Você escreveu uma classe que sabe ler um arquivo CSV e extrair qualquer uma de suas colunas. Ela é útil e você vai precisar dela em vários notebooks diferentes — não faz sentido copiar e colar o mesmo código toda vez.

> 💡 **Dica:** os `: str` e `: int` depois dos parâmetros abaixo são *type hints* — anotações opcionais que documentam o tipo esperado de cada argumento. O Python não as obriga, mas elas deixam o código mais fácil de entender.

In [14]:
# Lê um arquivo CSV e permite extrair colunas específicas
class ArquivoCSV:

  def __init__(self, caminho_arquivo: str):
    self.caminho_arquivo = caminho_arquivo
    self.linhas = self._ler_linhas()
    self.colunas = self._extrair_nomes_colunas()

  def _ler_linhas(self):
    with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
      return arquivo.readlines()

  def _extrair_nomes_colunas(self):
    return self.linhas[0].strip().split(',')

  def extrair_coluna(self, indice_coluna: int):
    valores = []
    for linha in self.linhas[1:]:  # [1:] pula a linha de cabeçalho
      valores.append(linha.strip().split(',')[indice_coluna])
    return valores

Um arquivo `banco.csv` para testar a classe:

In [15]:
conteudo_banco_csv = '''age,job,marital,education,default,balance,housing,loan
30,unemployed,married,primary,no,1787,no,no
33,services,married,secondary,no,4789,yes,yes
35,management,single,tertiary,no,1350,yes,no
30,management,married,tertiary,no,1476,yes,yes
59,blue-collar,married,secondary,no,0,yes,no
35,management,single,tertiary,no,747,no,no
36,self-employed,married,tertiary,no,307,yes,no
39,technician,married,secondary,no,147,yes,no
41,entrepreneur,married,tertiary,no,221,yes,no
43,services,married,primary,no,-88,yes,yes
'''

with open('banco.csv', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(conteudo_banco_csv)

> 💡 **Dica:** no Google Colab, o comando mágico `%%writefile nome_do_arquivo` faz exatamente a mesma coisa em uma única célula. Aqui usamos `open()` porque ela funciona em qualquer ambiente Python, não só no Colab.

In [16]:
arquivo_banco = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


### 2.2 Definição

Um módulo próprio é criado exatamente como o arquivo de dados acima: escrevemos o código em um arquivo `.py`. O Python encontra esse arquivo automaticamente quando ele está na mesma pasta do notebook — basta importá-lo pelo nome, sem a extensão.

> ⚠️ **Atenção:** a célula abaixo grava o código da classe em um arquivo `arquivo_csv.py` **a partir de uma string**, só para que este notebook consiga criar o módulo sozinho, do início ao fim. No seu dia a dia, você vai escrever `arquivo_csv.py` diretamente em um editor de código — e não gerá-lo a partir de outra célula.

In [17]:
codigo_do_modulo_csv = '''# Lê um arquivo CSV e permite extrair colunas específicas
class ArquivoCSV:

    def __init__(self, caminho_arquivo: str):
        self.caminho_arquivo = caminho_arquivo
        self.linhas = self._ler_linhas()
        self.colunas = self._extrair_nomes_colunas()

    def _ler_linhas(self):
        with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            return arquivo.readlines()

    def _extrair_nomes_colunas(self):
        return self.linhas[0].strip().split(',')

    def extrair_coluna(self, indice_coluna: int):
        valores = []
        for linha in self.linhas[1:]:
            valores.append(linha.strip().split(',')[indice_coluna])
        return valores
'''

with open('arquivo_csv.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_csv)

print('Módulo arquivo_csv.py criado.')

Módulo arquivo_csv.py criado.


### 2.3 Revisitando a motivação

Agora importamos a classe **do módulo** que acabamos de criar, em vez de tê-la definida diretamente no notebook:

In [18]:
from arquivo_csv import ArquivoCSV

arquivo_banco_modulo = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco_modulo.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


O resultado é idêntico ao da seção 2.1 — a diferença é que essa classe agora pode ser reaproveitada em **qualquer** notebook que esteja na mesma pasta, bastando importar.

Só ficam acessíveis os nomes que realmente existem dentro do módulo. Tentar usar um método que nunca foi definido gera um `AttributeError` (visto no Módulo 03):

In [19]:
try:
  soma = arquivo_banco_modulo._somar_saldos(coluna='balance')
except AttributeError as exc:
  print(f'Erro: {exc}')

Erro: 'ArquivoCSV' object has no attribute '_somar_saldos'


---

## 3. Pacotes

À medida que um projeto cresce, é comum acabar com vários módulos relacionados. Um **pacote** é uma pasta que agrupa esses módulos, permitindo organizá-los e importá-los a partir de um caminho só.

### 3.1 Motivação

Agora você também precisa processar arquivos de texto simples, como o conteúdo de uma notícia. Criamos outra classe para isso, parecida com a `ArquivoCSV`.

In [20]:
# Lê um arquivo de texto e permite extrair uma linha específica
class ArquivoTXT:

  def __init__(self, caminho_arquivo: str):
    self.caminho_arquivo = caminho_arquivo
    self.linhas = self._ler_linhas()

  def _ler_linhas(self):
    with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
      return arquivo.readlines()

  def extrair_linha(self, numero_linha: int):
    return self.linhas[numero_linha - 1].strip()

In [21]:
conteudo_noticia = '''Ready To Deploy lança módulo sobre organização de código em Python
O curso ensina a estruturar projetos maiores usando módulos e pacotes, seguindo as boas práticas mais usadas no mercado.
'''

with open('noticia.txt', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(conteudo_noticia)

In [22]:
arquivo_noticia = ArquivoTXT(caminho_arquivo='noticia.txt')

titulo = arquivo_noticia.extrair_linha(numero_linha=1)
print(titulo)

Ready To Deploy lança módulo sobre organização de código em Python


### 3.2 Definição

Um pacote é uma **pasta comum**, com um detalhe: tradicionalmente contém um arquivo `__init__.py` (pode estar vazio) que sinaliza ao Python que aquela pasta deve ser tratada como um pacote importável.

> 💡 **Dica:** a partir do Python 3.3, o interpretador também reconhece pastas sem `__init__.py` como pacotes "implícitos" (*namespace packages*). Ainda assim, incluir o arquivo continua sendo a prática mais comum e mais compatível entre versões.

Vamos criar um pacote chamado `arquivo` e mover os módulos `arquivo_csv.py` e `arquivo_txt.py` para dentro dele.

In [23]:
import os

os.makedirs('arquivo', exist_ok=True)

# Arquivo vazio que marca a pasta como um pacote
with open('arquivo/__init__.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write('')

print('Pacote "arquivo" criado.')

Pacote "arquivo" criado.


Reaproveitamos o mesmo código do módulo `ArquivoCSV`, escrito na seção 2.2, agora salvo dentro do pacote:

In [24]:
with open('arquivo/arquivo_csv.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_csv)

print('arquivo/arquivo_csv.py criado.')

arquivo/arquivo_csv.py criado.


In [25]:
codigo_do_modulo_txt = '''# Lê um arquivo de texto e permite extrair uma linha específica
class ArquivoTXT:

    def __init__(self, caminho_arquivo: str):
        self.caminho_arquivo = caminho_arquivo
        self.linhas = self._ler_linhas()

    def _ler_linhas(self):
        with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            return arquivo.readlines()

    def extrair_linha(self, numero_linha: int):
        return self.linhas[numero_linha - 1].strip()
'''

with open('arquivo/arquivo_txt.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_txt)

print('arquivo/arquivo_txt.py criado.')

arquivo/arquivo_txt.py criado.


### 3.3 Revisitando a motivação

Agora importamos os dois módulos **de dentro do pacote**, usando `.` para indicar o caminho pasta &#124; módulo:

In [26]:
from arquivo.arquivo_csv import ArquivoCSV
from arquivo.arquivo_txt import ArquivoTXT

In [27]:
arquivo_banco_pacote = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco_pacote.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


In [28]:
arquivo_noticia_pacote = ArquivoTXT(caminho_arquivo='noticia.txt')

titulo = arquivo_noticia_pacote.extrair_linha(numero_linha=1)
print(titulo)

Ready To Deploy lança módulo sobre organização de código em Python


> 💡 O resultado é o mesmo de antes, mas agora os dois módulos estão organizados dentro de uma única pasta — exatamente como bibliotecas reais (Pandas, NumPy...), que são pacotes com dezenas de módulos internos.

---

## 4. Baixando pacotes

Além da biblioteca padrão e dos módulos que você mesmo escreve, existe um universo de pacotes criados pela comunidade Python para praticamente qualquer tarefa.

### 4.1 PyPI

O **PyPI** (*Python Package Index*, [pypi.org](https://pypi.org/)) é o repositório oficial de pacotes Python. Nele você encontra bibliotecas para análise de dados, automação, desenvolvimento web, inteligência artificial e muito mais — cada uma com sua página, suas versões e sua documentação.

### 4.2 pip

O **pip** é a ferramenta oficial para instalar pacotes do PyPI direto do terminal (no Colab, em uma célula com `!` na frente, que executa um comando do sistema em vez de Python).

| Comando | O que faz |
| --- | --- |
| `pip install <pacote>` | Instala a versão mais recente do pacote |
| `pip install <pacote>==<versão>` | Instala uma versão específica |
| `pip freeze` | Lista todos os pacotes instalados e suas versões |
| `pip uninstall <pacote>` | Remove um pacote instalado |

In [29]:
!pip install requests==2.32.3

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.1.3 requires pyjwt>=2.10.0, but you have pyjwt 2.9.0 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.2.5 which is incompatible.
pinecone-plugin-assistant 1.8.0 requires packaging<25.0,>=24.2, but you have packaging 25.0 which is incompatible.
streamlit 1.38.0 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.
transformers 4.57.1 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.

In [30]:
!pip freeze

a2wsgi==1.10.10
absl-py==2.1.0
accelerate==1.10.1
acres==0.5.0
# Editable install with no version control (agent==0.0.1)
-e c:\users\schit\better-ai\my-app
agno==2.1.1
aiofiles==24.1.0
aiohappyeyeballs==2.4.6
aiohttp==3.11.12
aiohttp-retry==2.9.1
aiosignal==1.3.2
aiosmtplib==5.0.0
aiosqlite==0.21.0
alembic==1.17.2
altair==5.4.1
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.9.0
apache-airflow-core==3.1.3
apache-airflow-providers-common-compat==1.10.1
apache-airflow-providers-common-io==1.7.0
apache-airflow-providers-common-sql==1.30.1
apache-airflow-providers-smtp==2.4.1
apache-airflow-providers-standard==1.10.1
apache-airflow-task-sdk==1.1.3
appdirs==1.4.4
argcomplete==3.6.3
arxiv==2.2.0
asgiref==3.10.0
asttokens==2.4.1
astunparse==1.6.3
attrs==24.2.0
azure-core==1.35.1
azure-storage-blob==12.26.0
azure-storage-file-datalake==12.21.0
babel==2.17.0
backoff==2.2.1
bcrypt==4.3.0
beautifulsoup4==4.13.3
blinker==1.8.2
blockbuster==1.5.26
boto3==1.40.45
botocore==1.40.45
bs4=

> 💡 **Dica:** fixar a versão (`==2.32.3`) evita que uma atualização futura do pacote quebre um código que já estava funcionando — uma boa prática em projetos reais.

### 4.3 requests

O pacote [requests](https://pypi.org/project/requests/) facilita fazer requisições ao protocolo web **HTTP**, usado por praticamente todo serviço na internet.

**Exemplo:** você está automatizando o preenchimento de um cadastro e, a partir do CEP informado pelo cliente, precisa descobrir a rua, o bairro e a cidade. O [ViaCEP](https://viacep.com.br/) é um serviço público e gratuito para esse tipo de consulta.

In [31]:
import requests

resposta = requests.get('https://viacep.com.br/ws/01310930/json/')
print(f'Status code: {resposta.status_code}')

Status code: 200


Um `status_code` igual a `200` indica que a requisição foi bem-sucedida. O conteúdo retornado vem como texto, no formato **JSON** — um formato muito usado para troca de dados entre sistemas:

In [32]:
print(resposta.text)

{
  "cep": "01310-930",
  "logradouro": "Avenida Paulista",
  "complemento": "2100",
  "unidade": "Banco Safra S.A",
  "bairro": "Bela Vista",
  "localidade": "São Paulo",
  "uf": "SP",
  "estado": "São Paulo",
  "regiao": "Sudeste",
  "ibge": "3550308",
  "gia": "1004",
  "ddd": "11",
  "siafi": "7107"
}


O módulo `json`, também da biblioteca padrão, converte esse texto em um dicionário Python com `json.loads()`:

In [33]:
import json

endereco = json.loads(resposta.text)
print(endereco)

{'cep': '01310-930', 'logradouro': 'Avenida Paulista', 'complemento': '2100', 'unidade': 'Banco Safra S.A', 'bairro': 'Bela Vista', 'localidade': 'São Paulo', 'uf': 'SP', 'estado': 'São Paulo', 'regiao': 'Sudeste', 'ibge': '3550308', 'gia': '1004', 'ddd': '11', 'siafi': '7107'}


> 💡 O próprio `requests` já oferece um atalho para isso, sem precisar importar `json` manualmente: `resposta.json()`.

In [34]:
endereco = resposta.json()

print(f"Rua: {endereco['logradouro']}")
print(f"Bairro: {endereco['bairro']}")
print(f"Cidade: {endereco['localidade']} - {endereco['uf']}")

Rua: Avenida Paulista
Bairro: Bela Vista
Cidade: São Paulo - SP


> 💡 Praticamente todo pacote que conecta a uma API (redes sociais, bancos, sistemas de pagamento, o próprio ChatGPT...) segue essa mesma lógica: uma requisição HTTP que retorna dados em JSON.

---

## Resumo do Módulo

| Conceito | O que é | Sintaxe |
| --- | --- | --- |
| Importar um módulo inteiro | Traz todo o conteúdo, acessado com prefixo | `import modulo` |
| Importar nomes específicos | Traz só o que foi pedido, sem prefixo | `from modulo import nome` |
| Apelidar na importação | Renomeia o módulo ou o nome importado | `import modulo as apelido` |
| Módulo | Um arquivo `.py` com código reutilizável | `arquivo_csv.py` |
| Pacote | Uma pasta com módulos relacionados e um `__init__.py` | `arquivo/arquivo_csv.py` |
| PyPI | Repositório oficial de pacotes de terceiros | [pypi.org](https://pypi.org/) |
| pip | Ferramenta para instalar, listar e remover pacotes | `pip install`, `pip freeze`, `pip uninstall` |

Pacotes vistos neste módulo: `random`, `math`, `time`, `datetime`, `os`, `json` (biblioteca padrão) e `requests` (de terceiros, via PyPI).